In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_metal_coordination_sites)=
# get_metal_coordination_sites

*Recognizing metal and ligand atoms before measuring geometry.*

This experimental chemical profile reproduces the pinned ProLIF 2.2.2 MetalDonor SMARTS. The general tool belongs in physchem; no coordinates, bonds or oxidation states are inferred. RDKit supplies graph matching, while ProLIF is a reference rather than an executed dependency.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_metal_coordination_sites` for arguments, outputs and explicit failures.
:::

## Basic usage

Inspect zinc, hydrated oxygen and nitrogen sites independently of geometry.

In [2]:
import molsysmt as msm
from rdkit import Chem
molsys = Chem.MolFromSmiles('[Zn+2].O.N.[Na+]')
sites = msm.physchem.get_metal_coordination_sites(molsys)
print(sites['metal_atom_indices'], sites['ligand_atom_indices'])

[0] [1 2]


## Chemical restrictions

The set includes Ca, Cd, Co, Cu, Fe, Mg, Mn, Ni and Zn; Na and K are outside it. Neutral oxygen, restricted nitrogen and negative atoms can match ligand roles. Do not assume every N or S matches.

In [3]:
molsys = Chem.MolFromSmiles('[Zn+2].CC(=O)N.C[N+](C)(C)C.[S-].c1ccncc1')
sites = msm.physchem.get_metal_coordination_sites(molsys)
print([(i, molsys.GetAtomWithIdx(int(i)).GetSymbol()) for i in sites['ligand_atom_indices']])

[(np.int64(3), 'O'), (np.int64(10), 'S'), (np.int64(14), 'N')]


## Selecting source atoms

Full graph interpretation precedes filtering. Selecting an amide nitrogen alone does not turn it into an eligible ligand.

In [4]:
selected = msm.physchem.get_metal_coordination_sites(molsys, selection=[0, 4])
print(selected['metal_atom_indices'], selected['ligand_atom_indices'])
empty = msm.physchem.get_metal_coordination_sites(Chem.MolFromSmiles('CC'))
print(empty['metal_atom_indices'].shape, empty['ligand_atom_indices'].shape)

[0] []


(0,) (0,)


## Interpreting biological chemistry

Zinc sites can involve histidine and carboxylate coordination. The following fragments illustrate chemical recognition only; they are not a protein coordination geometry or an affinity model.

In [5]:
molsys = Chem.MolFromSmiles('[Zn+2].CC(=O)[O-].c1cc[nH]c1')
print(msm.physchem.get_metal_coordination_sites(molsys)['ligand_atom_indices'])

[3 4]


## Using native chemistry and state selection

Public inputs are form agnostic. Choose the reference state or its explicit index; structure-assigned states must resolve one state. Dative links remain separate from the recognized covalent graph.

In [6]:
molsys = msm.convert(molsys, to_form='molsysmt.MolSys')
print(msm.physchem.get_metal_coordination_sites(molsys, chemical_state=0)['metal_atom_indices'])

[0]


## Inspecting evidence and bounded matching

The default match cap is 100,000 per full-source SMARTS. Increase it deliberately for larger systems; exceeded caps raise. Incomplete chemistry also raises unless connectivity completeness is explicitly assumed; the assumption does not repair assignments.

In [7]:
sites = msm.physchem.get_metal_coordination_sites(molsys, max_matches=1000)
print(sites['method'], sites['method_reference']['version'], sites['software'])
print([item['roles'] for item in sites['attribution']['items']])

smarts_metal_ligand 2.2.2 {'molsysmt': '0.21.0+606.ga03eb4bf6', 'rdkit': '2025.09.5'}
[['reference_implementation'], ['executed_software'], ['executed_software']]


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Sparse interaction results <user-tools-interactions-result>`
- {func}`molsysmt.Interactions.query` — querying observations by actual atom and structure indices.
:::